# 🎯 Arabic Pronunciation Assessment – Production Backend

**State-of-the-art Pipeline** (fest konfiguriert, keine Alternativen):

| Stufe | Tool | Zweck |
|---|---|---|
| VAD / Trim | **Silero VAD 5** | Stille am Anfang/Ende entfernen |
| ASR | **wav2vec 2.0 XLSR-53 Arabic** | Beste offene Arabisch-Erkennung |
| Alignment | **`torchaudio.functional.forced_align`** | CUDA-beschleunigte CTC-Alignment |
| Scoring | **GOP** (mittlere Log-Prob pro Buchstabe) | 0 – 100 pro Buchstabe |
| Serving | **FastAPI + Uvicorn**, Pydantic-Validierung | Getypter Endpoint mit Limits |
| Tunnel | **ngrok** (nativer WebSocket-Support) | Öffentliche `https://…ngrok-free.app`-URL |

**Ausführung in Colab:**
1. `Runtime → Change runtime type → T4 GPU`
2. Einmalig: 🔑 **Secrets** links → Add → `NGROK_TOKEN` = dein Token von [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)
3. `Runtime → Run all`
4. Letzte Zelle druckt die URL für die App.

In [ ]:
%pip install -q -U transformers pydub nest_asyncio python-multipart
%pip install -q -U fastapi 'uvicorn[standard]'
%pip install -q -U silero-vad

# scipy bewusst NICHT hier upgraden: die Colab-Basis liefert scipy vorinstalliert.
# Ein "pip install -U scipy" waehrend einer laufenden Session verursacht in Colab
# einen Cython-ABI-Mismatch (siehe: scipy._cyutility / __Pyx__Import ImportError),
# weil numpy/scipy-Extensions bereits im Speicher liegen.

!apt-get -qq install -y ffmpeg > /dev/null

import importlib
for mod in ("transformers", "pydub", "fastapi", "uvicorn", "silero_vad", "torch", "torchaudio", "scipy"):
    m = importlib.import_module(mod)
    print(f"  ✅ {mod:14s} {getattr(m, '__version__', '?')}")
print("✅ Alle Abhängigkeiten importierbar.")

In [ ]:
import io, os, re, time, subprocess, threading, urllib.request, unicodedata
from typing import List, Dict, Any

import numpy as np
import torch
import torchaudio.functional as AF
from pydub import AudioSegment
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from silero_vad import load_silero_vad, get_speech_timestamps

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SR = 16000
print(f"✅ Device: {device}, torch {torch.__version__}")

In [ ]:
ASR_MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-arabic"

# fp16 nur auf GPU: ~2x schnellere Inferenz, kein Qualitaetsverlust bei CTC.
USE_FP16 = device.type == "cuda"
DTYPE    = torch.float16 if USE_FP16 else torch.float32

print("Lade ASR-Modell …")
asr_processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_ID)
asr_model     = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_ID).to(device=device, dtype=DTYPE).eval()
ASR_VOCAB    = asr_processor.tokenizer.get_vocab()
ASR_BLANK_ID = asr_model.config.pad_token_id
print(f"  ✅ wav2vec2 XLSR-53 Arabic  ({len(ASR_VOCAB)} tokens, blank={ASR_BLANK_ID}, dtype={DTYPE})")

print("Lade Silero VAD …")
vad_model = load_silero_vad()
print("  ✅ Silero VAD 5")

# Warm-up mit realistischer 2s-Laenge, damit spaeter kein CUDA-JIT die erste Anfrage bremst.
with torch.inference_mode():
    _ = asr_model(torch.zeros(1, 2 * SR, device=device, dtype=DTYPE)).logits
print("✅ Modelle geladen und aufgewärmt.")

In [ ]:
from scipy.signal import butter, sosfiltfilt

# Signalkette vor dem ASR (alle Schritte sind linear, phasentreu oder nicht-neuronal
# -> beruehren die spektrale Signatur arabischer Gutturale/Emphatika/Frikative nicht):
#   decode  -> HP80Hz  -> RMS-Norm -> gentle_trim (VAD) -> Kontext-Pad
_HPF_SOS = butter(2, 80.0, btype="highpass", fs=SR, output="sos")

def decode_audio(raw: bytes) -> np.ndarray:
    """Beliebiges Audioformat -> 16 kHz mono float32."""
    seg = AudioSegment.from_file(io.BytesIO(raw))
    seg = seg.set_frame_rate(SR).set_channels(1).set_sample_width(2)
    return np.asarray(seg.get_array_of_samples(), dtype=np.float32) / 32768.0

def _highpass(audio: np.ndarray) -> np.ndarray:
    """Nullphasiger Butterworth @ 80 Hz: entfernt DC-Offset, Handling-Rumpeln,
    Netzbrummen (50/60 Hz). Liegt unterhalb jeder Sprachformantenergie."""
    return sosfiltfilt(_HPF_SOS, audio).astype(np.float32)

def _normalize_level(audio: np.ndarray, target_dbfs: float = -20.0) -> np.ndarray:
    """RMS-Normalisierung auf konsistentes Pegel -> Silero-VAD-Schwelle wird reproduzierbar,
    und wav2vec2s eingebautes do_normalize=True bekommt ein saubereres Zero-Mean/Unit-Var-Ziel."""
    rms = float(np.sqrt(np.mean(audio ** 2)))
    if rms < 1e-6:
        return audio
    gain = 10.0 ** ((target_dbfs - 20.0 * np.log10(rms)) / 20.0)
    out  = audio * gain
    peak = float(np.max(np.abs(out)))
    if peak > 0.99:
        out = out / peak * 0.99
    return out.astype(np.float32)

def gentle_trim(audio: np.ndarray, pad_ms: int = 120) -> np.ndarray:
    """Nur führende/nachlaufende lange Stille entfernen. Zwischenpausen bleiben."""
    segs = get_speech_timestamps(torch.from_numpy(audio), vad_model,
                                 sampling_rate=SR, threshold=0.35)
    if not segs:
        return audio
    pad = int(pad_ms * SR / 1000)
    start = max(0, segs[0]["start"] - pad)
    end   = min(len(audio), segs[-1]["end"] + pad)
    return audio[start:end]

def _pad_context(audio: np.ndarray, ms: int = 250) -> np.ndarray:
    """Wav2vec2-Transformer sieht pro Frame ein bidirektionales Kontextfenster (~200 ms).
    Kurze Woerter (2-3 Buchstaben) verlieren sonst am Anfang/Ende Kontextframes und werden
    systematisch schlechter erkannt. Silence-Padding kostet keine Latenz und keine Genauigkeit."""
    pad = np.zeros(int(ms * SR / 1000), dtype=np.float32)
    return np.concatenate([pad, audio, pad])

def preprocess(raw: bytes) -> np.ndarray:
    audio = decode_audio(raw)
    audio = _highpass(audio)
    audio = _normalize_level(audio)
    audio = gentle_trim(audio)
    audio = _pad_context(audio)
    return audio

In [ ]:
# Nur klassisches Tashkeel entfernen. Hamza-Formen (أ إ آ ؤ ئ) bleiben als eigene Buchstaben erhalten.
_TASHKEEL = set("ًٌٍَُِّْٰ")

def strip_diacritics(text: str) -> str:
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if c not in _TASHKEEL))

# Positionsabhaengige Aequivalenzen (Anfang/Ende) fuer Posterior-Bewertung.
_START_EQUIV = {ch: "اأإآ" for ch in "اأإآ"}
_END_EQUIV   = {"ة": "ةه", "ه": "هة",
                "ى": "ىيا", "ي": "يى"}

# Linguistisch belegte Verwechslungen fuer den LLR-Test.
# Quellen: Al-Ani (1970) "Arabic Phonology"; Newman (2013);
# Standard-DaF/L2-Arabisch-Fehlerkataloge; Kinder-L1-Erwerbsstudien.
_CONFUSABLES: Dict[str, str] = {
    "ت": "طثد",
    "ث": "تسذف",
    "ح": "هخع",
    "خ": "حغك",
    "د": "تضذ",
    "ذ": "دزثظ",
    "ر": "لغ",
    "ز": "ذسظ",
    "س": "صثزش",
    "ش": "سج",
    "ص": "سض",
    "ض": "دظص",
    "ط": "تضد",
    "ظ": "زذض",
    "ع": "ءأاه",
    "غ": "خقر",
    "ق": "كغخ",
    "ك": "قخج",
    "ل": "ر",
    "ه": "حة",
    "ء": "ع",
    "ج": "شك",
}

def _equiv_ids(ch: str, pos: int, total: int) -> List[int]:
    if pos == 0 and ch in _START_EQUIV:
        alts = _START_EQUIV[ch]
    elif pos == total - 1 and ch in _END_EQUIV:
        alts = _END_EQUIV[ch]
    else:
        alts = ch
    ids = [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]
    return ids or [ASR_VOCAB[ch]]

def _confuse_ids(ch: str) -> List[int]:
    alts = _CONFUSABLES.get(ch, "")
    return [ASR_VOCAB[c] for c in alts if c in ASR_VOCAB]

# Umkehr-Map: Token-ID -> Buchstabe, fuer error_hint.
_ID_TO_CHAR = {tid: c for c, tid in ASR_VOCAB.items()}

def encode_target(word: str) -> List[int]:
    ids: List[int] = []
    for ch in word:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise ValueError(f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        ids.append(tid)
    return ids

@torch.inference_mode()
def run_asr(audio: np.ndarray):
    inputs = asr_processor(audio, sampling_rate=SR, return_tensors="pt", padding=True)
    input_values = inputs.input_values.to(device=device, dtype=DTYPE)
    logits = asr_model(input_values).logits
    # log_softmax stabil in fp32, torchaudio.forced_align verlangt float32 CPU.
    log_probs = torch.log_softmax(logits.float(), dim=-1).cpu()
    transcription = asr_processor.batch_decode(log_probs.argmax(dim=-1))[0]
    return log_probs, transcription

def _runs_of_non_blank(tokens: List[int]) -> List[List[int]]:
    runs: List[List[int]] = []
    current: List[int] = []
    last: int = -1
    for t, tok in enumerate(tokens):
        if tok == ASR_BLANK_ID:
            if current: runs.append(current); current = []
            last = -1
        elif tok != last:
            if current: runs.append(current)
            current = [t]; last = tok
        else:
            current.append(t)
    if current: runs.append(current)
    return runs

# Kalibrierungskonstante: LLR=0 -> 50, LLR=+1 -> ~88, LLR=-1 -> ~12.
_LLR_K = 2.0

def _sigmoid(x: float) -> float:
    return 1.0 / (1.0 + float(np.exp(-x)))

def gop_score(log_probs: torch.Tensor, target_word: str) -> List[Dict[str, Any]]:
    target_ids = encode_target(target_word)
    if not target_ids:
        return []
    if log_probs.shape[1] < len(target_ids):
        raise ValueError("Aufnahme zu kurz für dieses Wort.")
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    total_len = len(target_word)
    results: List[Dict[str, Any]] = []
    for i, ch in enumerate(target_word):
        if i >= len(runs):
            results.append({"label": ch, "score": 0.0, "confidence": 0.0,
                            "llr": -5.0, "error_hint": None})
            continue

        frames   = runs[i]
        lp_frame = log_probs[0, frames]  # [F, V]

        # 1) Posterior-Score (klassisches GOP, positionsbewusst).
        equiv_ids  = _equiv_ids(ch, i, total_len)
        target_lp  = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
        post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
        conf       = float(np.exp(target_lp))

        # 2) LLR gegen dokumentierte Verwechslungen (Anti-Modell).
        confuse_ids = _confuse_ids(ch)
        if confuse_ids:
            per_frame_conf = lp_frame[:, confuse_ids]
            best_conf_lp   = per_frame_conf.max(dim=-1).values.mean().item()
            llr            = target_lp - best_conf_lp
            llr_score      = _sigmoid(_LLR_K * llr) * 100.0
            # Nur melden wenn Verwechslung staerker als Ziel.
            if llr < 0:
                best_col   = int(per_frame_conf.mean(dim=0).argmax().item())
                hint_id    = confuse_ids[best_col]
                error_hint = _ID_TO_CHAR.get(hint_id)
            else:
                error_hint = None
        else:
            llr, llr_score, error_hint = 5.0, 100.0, None

        # 3) Kombination: 40 % Posterior + 60 % LLR (LLR ist informativer).
        final = 0.4 * post_score + 0.6 * llr_score
        results.append({
            "label": ch,
            "score": float(np.clip(final, 0, 100)),
            "confidence": conf,
            "llr": float(llr),
            "error_hint": error_hint,
        })
    return results

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional, Iterator
import asyncio, json
from collections import deque
from datetime import datetime

MAX_AUDIO_BYTES     = 3 * 1024 * 1024
MAX_AYAH_AUDIO_BYTES = 8 * 1024 * 1024
MIN_SAMPLES     = int(0.15 * SR)

# Cloudflared Quick Tunnel killt WS-Verbindungen nach ~60-100s Inaktivitaet.
# Wir senden alle 20s einen App-Level-Ping, damit die Verbindung fuer die
# ganze Kind-Session offen bleibt (sonst Reconnect-Cost pro Wort ~500ms).
WS_KEEPALIVE_SEC = 20

# Zwischen Wort-Frames im Ayah-Stream: minimaler Delay, damit die UI
# das progressive Einfaerben visuell wahrnimmt statt "alles auf einmal".
WORD_STREAM_DELAY_SEC = 0.035

# ----- Timing-Log (In-Memory Ring + Datei, ueber GET /logs + !tail abrufbar) -----
_LOG_BUF: deque = deque(maxlen=200)
_LOG_FILE = "/content/backend.log"
try:
    # datei zuruecksetzen bei jedem Cell-Rerun
    open(_LOG_FILE, "w").close()
except Exception:
    _LOG_FILE = "/tmp/backend.log"
    try: open(_LOG_FILE, "w").close()
    except Exception: pass

def _log(event: str, **kv):
    """Struktur-Log: Colab-Cell + In-Memory-Ring + Datei /content/backend.log.
    Ansicht:  !tail -n 40 /content/backend.log   oder   <BACKEND_URL>/logs"""
    entry = {"ts": datetime.utcnow().isoformat(timespec="milliseconds") + "Z",
             "event": event, **kv}
    _LOG_BUF.append(entry)
    kv_str = " ".join(f"{k}={v}" for k, v in kv.items())
    line = f"[{entry['ts']}] {event}  {kv_str}"
    print(line, flush=True)
    try:
        with open(_LOG_FILE, "a") as fh:
            fh.write(line + "\n"); fh.flush()
    except Exception:
        pass

class Unit(BaseModel):
    label: str
    score: float
    confidence: float
    llr: Optional[float] = None
    error_hint: Optional[str] = None

class AssessResponse(BaseModel):
    target: str
    transcription: str
    units: List[Unit]
    total: float
    duration_ms: int

app = FastAPI(title="Arabic Pronunciation API", version="1.3.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

# Middleware: jeder HTTP-Request wird geloggt (auch /health). Damit sieht man
# im /logs sofort, ob der Client ueberhaupt am Backend ankommt.
@app.middleware("http")
async def _request_logger(request, call_next):
    t0 = time.perf_counter()
    resp = await call_next(request)
    # /logs selbst NICHT loggen, sonst rekursive Flut beim Auto-Refresh.
    if not request.url.path.startswith("/logs"):
        _log("http",
             method=request.method,
             path=request.url.path,
             status=resp.status_code,
             ms=int((time.perf_counter() - t0) * 1000),
             client=request.client.host if request.client else "?")
    return resp

def _score_word(raw: bytes, target: str) -> Dict[str, Any]:
    """Synchrone Bewertungs-Pipeline. Wird sowohl vom HTTP- als auch vom WS-Endpoint aufgerufen,
    damit die Bewertungsqualitaet identisch bleibt."""
    if len(raw) > MAX_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungültig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    target_clean = strip_diacritics(target)
    log_probs, transcription = run_asr(wav)
    units = gop_score(log_probs, target_clean)
    total = float(np.mean([u["score"] for u in units])) if units else 0.0
    return {
        "target": target_clean,
        "transcription": transcription,
        "units": units,
        "total": total,
    }

# ---------------------------------------------------------------------------
# Ayah-Modus: eine ganze Ayah (mehrere Woerter) in EINEM ASR-Forward-Pass
# scoren und pro Wort progressiv an den Client streamen.
#
# Design:
#   1. Alle Woerter der Ayah werden zu EINEM Buchstaben-Ziel konkateniert.
#      forced_align liefert damit ein globales, konsistentes Alignment ueber
#      die gesamte Rezitation - genauer als N unabhaengige Wort-Alignments,
#      weil Uebergaenge zwischen Woertern (Waslah, Idghaam) mitmodelliert werden.
#   2. Aus den Buchstaben-Runs werden per Wort-Grenzen-Map wieder Wort-Scores
#      aggregiert (Mittelwert der Buchstaben-Scores + Mindestwert-Penalty
#      falls einzelne Buchstaben stark abfallen).
#   3. Der WS-Handler yielded die Wort-Ergebnisse einzeln mit minimalem Delay,
#      damit der Client die Einfaerbung wie eine Live-Auswertung rendert.
#
# Woerter mit Tashkeel werden per strip_diacritics gecleant; Hamza-Formen
# bleiben erhalten (siehe _TASHKEEL). Wenn die Aufnahme zu kurz ist, um alle
# Buchstaben zu enthalten, bekommen fehlende Runs Score 0 - der Client kann
# das als "abgeschnitten" darstellen.
# ---------------------------------------------------------------------------

def _score_ayah_streamed(raw: bytes, ayah_text: str) -> Iterator[Dict[str, Any]]:
    if len(raw) > MAX_AYAH_AUDIO_BYTES:
        raise HTTPException(413, f"Audio > {MAX_AYAH_AUDIO_BYTES // 1024} KB.")
    if not raw:
        raise HTTPException(400, "Leere Audiodatei.")

    # 1) Ayah in Woerter zerlegen (Whitespace-basiert). Leere Tokens verwerfen.
    raw_words = [w for w in ayah_text.split() if w.strip()]
    if not raw_words:
        raise HTTPException(400, "Ayah-Text leer.")

    words_clean: List[str] = []
    for w in raw_words:
        cw = strip_diacritics(w)
        if cw:
            words_clean.append(cw)
    if not words_clean:
        raise HTTPException(400, "Ayah-Text enthaelt keine bewertbaren Zeichen.")

    # 2) Kompletter Buchstaben-Stream + Wort-Grenzen-Map [start, end).
    all_chars: List[str] = []
    word_spans: List[tuple] = []
    for w in words_clean:
        s = len(all_chars)
        all_chars.extend(list(w))
        word_spans.append((s, len(all_chars)))

    # 3) Ziel-IDs; unbekannte Zeichen -> Fehler mit klarer Meldung.
    target_ids: List[int] = []
    for ch in all_chars:
        tid = ASR_VOCAB.get(ch)
        if tid is None:
            raise HTTPException(400, f"Zeichen {ch!r} nicht im ASR-Vokabular.")
        target_ids.append(tid)

    # 4) Preprocess + ASR (ein Forward-Pass fuer die ganze Ayah).
    t_pre = time.perf_counter()
    try:
        wav = preprocess(raw)
    except Exception as e:
        raise HTTPException(400, f"Audio ungueltig: {e}")
    if wav.size < MIN_SAMPLES:
        raise HTTPException(400, "Aufnahme zu kurz.")
    dt_pre = int((time.perf_counter() - t_pre) * 1000)

    t_asr = time.perf_counter()
    log_probs, transcription = run_asr(wav)
    dt_asr = int((time.perf_counter() - t_asr) * 1000)

    if log_probs.shape[1] < len(target_ids):
        raise HTTPException(400, "Aufnahme zu kurz fuer diese Ayah.")

    t_align = time.perf_counter()
    targets = torch.tensor([target_ids], dtype=torch.int32)
    aligned, _ = AF.forced_align(log_probs, targets, blank=ASR_BLANK_ID)
    runs = _runs_of_non_blank(aligned[0].tolist())
    dt_align = int((time.perf_counter() - t_align) * 1000)

    # 5) Start-Frame: gibt dem Client die Wort-Anzahl + Transkription zur Anzeige.
    yield {
        "kind": "start",
        "words_count": len(words_clean),
        "transcription": transcription,
    }

    t_score = time.perf_counter()
    per_word_scores: List[float] = []
    for wi, (start_c, end_c) in enumerate(word_spans):
        word = words_clean[wi]
        word_len = end_c - start_c
        char_units: List[Dict[str, Any]] = []

        for local_i, char_global_i in enumerate(range(start_c, end_c)):
            ch = all_chars[char_global_i]
            if char_global_i >= len(runs):
                char_units.append({
                    "label": ch, "score": 0.0, "confidence": 0.0,
                    "llr": -5.0, "error_hint": None,
                })
                continue

            frames = runs[char_global_i]
            lp_frame = log_probs[0, frames]

            # Positionsbewusstsein: Anfang/Ende gilt PRO WORT, nicht pro Ayah.
            equiv_ids = _equiv_ids(ch, local_i, word_len)
            target_lp = lp_frame[:, equiv_ids].max(dim=-1).values.mean().item()
            post_score = float(np.clip((target_lp + 3.0) / 3.0 * 100, 0, 100))
            conf = float(np.exp(target_lp))

            confuse_ids = _confuse_ids(ch)
            if confuse_ids:
                per_frame_conf = lp_frame[:, confuse_ids]
                best_conf_lp = per_frame_conf.max(dim=-1).values.mean().item()
                llr = target_lp - best_conf_lp
                llr_score = _sigmoid(_LLR_K * llr) * 100.0
                if llr < 0:
                    best_col = int(per_frame_conf.mean(dim=0).argmax().item())
                    hint_id = confuse_ids[best_col]
                    error_hint = _ID_TO_CHAR.get(hint_id)
                else:
                    error_hint = None
            else:
                llr, llr_score, error_hint = 5.0, 100.0, None

            final = 0.4 * post_score + 0.6 * llr_score
            char_units.append({
                "label": ch,
                "score": float(np.clip(final, 0, 100)),
                "confidence": conf,
                "llr": float(llr),
                "error_hint": error_hint,
            })

        # Wort-Score: Mittelwert MIT Min-Penalty. Ein katastrophaler Buchstabe
        # zieht das Ergebnis staerker runter als reines Averaging, aber nicht
        # so hart, dass ein sonst gutes Wort komplett rot wird.
        char_scores = [u["score"] for u in char_units]
        mean_s = float(np.mean(char_scores)) if char_scores else 0.0
        min_s  = float(np.min(char_scores))  if char_scores else 0.0
        word_score = float(np.clip(0.75 * mean_s + 0.25 * min_s, 0, 100))
        per_word_scores.append(word_score)

        yield {
            "kind": "word",
            "word_idx": wi,
            "target": word,
            "score": word_score,
            "units": char_units,
        }

    total = float(np.mean(per_word_scores)) if per_word_scores else 0.0
    dt_score = int((time.perf_counter() - t_score) * 1000)
    yield {
        "kind": "done",
        "total": total,
        "words_count": len(words_clean),
        "timings": {
            "audio_bytes": len(raw),
            "audio_samples": int(wav.size),
            "audio_ms": int(wav.size * 1000 / SR),
            "preprocess_ms": dt_pre,
            "asr_ms": dt_asr,
            "align_ms": dt_align,
            "score_ms": dt_score,
        },
    }

@app.get("/health")
def health():
    return {"status": "ok", "device": str(device),
            "asr_model": ASR_MODEL_ID, "vad": "Silero VAD 5",
            "fp16": USE_FP16,
            "endpoints": ["/assess (HTTP)", "/stream (WebSocket)", "/logs"]}

@app.get("/logs")
def get_logs(n: int = 50, fmt: str = "html"):
    """Letzte N Log-Eintraege. Aus dem Handy-Browser:  <BACKEND_URL>/logs?n=40
    HTML mit Auto-Refresh (Standard) oder ?fmt=json fuer maschinell."""
    n = max(1, min(int(n), _LOG_BUF.maxlen or 200))
    entries = list(_LOG_BUF)[-n:][::-1]  # neueste oben
    if fmt == "json":
        return {"count": len(entries), "entries": entries}
    from fastapi.responses import HTMLResponse
    if not entries:
        rows = "<tr><td colspan='2' style='color:#94a3b8'>Noch keine Requests aufgezeichnet.</td></tr>"
    else:
        keys = ["ts", "event"] + sorted({k for e in entries for k in e if k not in ("ts", "event")})
        head = "".join(f"<th>{k}</th>" for k in keys)
        body_rows = []
        for e in entries:
            cells = "".join(f"<td>{e.get(k, '')}</td>" for k in keys)
            body_rows.append(f"<tr>{cells}</tr>")
        rows = f"<tr>{head}</tr>" + "".join(body_rows)
    html = f"""<!doctype html><html><head><meta charset='utf-8'>
<meta name='viewport' content='width=device-width,initial-scale=1'>
<meta http-equiv='refresh' content='2'>
<title>Backend-Logs</title>
<style>
body{{font-family:-apple-system,Segoe UI,Roboto,sans-serif;margin:0;padding:12px;background:#0f172a;color:#e2e8f0}}
h1{{font-size:15px;margin:0 0 8px}}
table{{width:100%;border-collapse:collapse;font-size:11px;font-family:ui-monospace,Menlo,Consolas,monospace}}
th{{text-align:left;padding:6px 8px;background:#1e293b;color:#93c5fd;position:sticky;top:0}}
td{{padding:5px 8px;border-top:1px solid #1e293b;color:#e2e8f0;white-space:nowrap}}
tr:nth-child(even) td{{background:#0b1220}}
.small{{color:#64748b;font-size:11px}}
</style></head><body>
<h1>Backend-Logs <span class='small'>· auto-refresh 2s · n={n}</span></h1>
<table>{rows}</table>
</body></html>"""
    return HTMLResponse(content=html)

@app.post("/assess", response_model=AssessResponse)
def assess(audio: UploadFile = File(...), target: str = Form(...)):
    target = target.strip()
    if not target:
        raise HTTPException(400, "Zielwort fehlt.")
    t0 = time.perf_counter()
    raw = audio.file.read(MAX_AUDIO_BYTES + 1)
    try:
        result = _score_word(raw, target)
    except ValueError as e:
        raise HTTPException(400, str(e))
    result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
    return AssessResponse(units=[Unit(**u) for u in result["units"]], **{
        k: v for k, v in result.items() if k != "units"
    })

@app.websocket("/stream")
async def stream_ws(ws: WebSocket):
    """Persistente Session pro Kind. Zwei Modi ueber dasselbe WS.

    A) Einzelwort-Modus (bestehend, Play-Modus):
       1. Client -> Server: {"target": "kitab"} (Text-Frame)
       2. Client -> Server: Binaer-Frame mit Audio
       3. Server -> Client: AssessResponse-JSON

    B) Ayah-Modus (Quran-Reading, progressives Per-Wort-Streaming):
       1. Client -> Server: {"mode": "ayah", "ayah": "bism allah..."}
       2. Client -> Server: Binaer-Frame mit Audio
       3. Server -> Client (Serie):
            {"kind":"start","words_count":N,"transcription":"..."}
            {"kind":"word","word_idx":0,"target":"...","score":..,"units":[..]}
            ...
            {"kind":"done","total":..,"duration_ms":..}

    Zusaetzlich sendet der Server alle WS_KEEPALIVE_SEC Sekunden
    {"ping": true}, damit Cloudflared die Verbindung nicht als idle killt.
    """
    await ws.accept()
    _log("ws_open", client=ws.client.host if ws.client else "?")

    async def keepalive():
        try:
            while True:
                await asyncio.sleep(WS_KEEPALIVE_SEC)
                await ws.send_json({"ping": True})
        except Exception:
            return

    ka_task = asyncio.create_task(keepalive())

    try:
        while True:
            ctrl = json.loads(await ws.receive_text())
            mode = str(ctrl.get("mode", "word")).lower()

            if mode == "ayah":
                ayah = str(ctrl.get("ayah", "")).strip()
                if not ayah:
                    await ws.send_json({"error": "Ayah-Text fehlt."})
                    continue
                t_ctrl = time.perf_counter()
                msg = await ws.receive()
                if "bytes" not in msg or msg["bytes"] is None:
                    await ws.send_json({"error": "Erwartete Binaerdaten (Audio)."})
                    continue
                raw: bytes = msg["bytes"]
                dt_bytes_ms = int((time.perf_counter() - t_ctrl) * 1000)
                t0 = time.perf_counter()
                try:
                    frames = await asyncio.to_thread(
                        lambda: list(_score_ayah_streamed(raw, ayah))
                    )
                except HTTPException as e:
                    _log("ayah_err", detail=e.detail, bytes=len(raw))
                    await ws.send_json({"error": e.detail}); continue
                except ValueError as e:
                    _log("ayah_err", detail=str(e), bytes=len(raw))
                    await ws.send_json({"error": str(e)}); continue
                except Exception as e:
                    _log("ayah_err", detail=str(e), bytes=len(raw))
                    await ws.send_json({"error": f"Serverfehler: {e}"}); continue

                dt_compute = int((time.perf_counter() - t0) * 1000)
                t_stream = time.perf_counter()
                for f in frames:
                    if f.get("kind") == "done":
                        f["duration_ms"] = int((time.perf_counter() - t0) * 1000)
                        f.setdefault("timings", {})["bytes_recv_ms"] = dt_bytes_ms
                    await ws.send_json(f)
                    if f.get("kind") == "word":
                        await asyncio.sleep(WORD_STREAM_DELAY_SEC)
                dt_stream = int((time.perf_counter() - t_stream) * 1000)

                # Zusammenfassung fuer /logs und Colab-Cell.
                done = next((f for f in frames if f.get("kind") == "done"), {})
                t = done.get("timings", {})
                _log("ayah",
                     words=done.get("words_count"),
                     total=round(done.get("total", 0), 1),
                     bytes=len(raw),
                     audio_ms=t.get("audio_ms"),
                     recv=dt_bytes_ms,
                     pre=t.get("preprocess_ms"),
                     asr=t.get("asr_ms"),
                     align=t.get("align_ms"),
                     score=t.get("score_ms"),
                     stream=dt_stream,
                     compute=dt_compute)
                continue

            # --- Einzelwort-Modus (backwards compatible) ---
            target = str(ctrl.get("target", "")).strip()
            if not target:
                await ws.send_json({"error": "Zielwort fehlt."})
                continue
            msg = await ws.receive()
            if "bytes" not in msg or msg["bytes"] is None:
                await ws.send_json({"error": "Erwartete Binaerdaten (Audio)."})
                continue
            raw: bytes = msg["bytes"]
            t0 = time.perf_counter()
            try:
                result = await asyncio.to_thread(_score_word, raw, target)
            except HTTPException as e:
                await ws.send_json({"error": e.detail})
                continue
            except ValueError as e:
                await ws.send_json({"error": str(e)})
                continue
            except Exception as e:
                await ws.send_json({"error": f"Serverfehler: {e}"})
                continue
            result["duration_ms"] = int((time.perf_counter() - t0) * 1000)
            await ws.send_json(result)
    except WebSocketDisconnect:
        _log("ws_close", reason="disconnect")
        return
    except Exception as e:
        _log("ws_close", reason=f"exception:{e}")
        try: await ws.send_json({"error": f"Serverfehler: {e}"})
        except Exception: pass
    finally:
        ka_task.cancel()

print("✅ API definiert:  GET /health   POST /assess   WS /stream  (Wort+Ayah, Keep-Alive 20s)")

In [ ]:
import uvicorn, nest_asyncio
nest_asyncio.apply()

PORT = 8000

# --- ngrok statt Cloudflared: nativer, stabiler WebSocket-Support ---
# Vorbereitung EINMALIG in Colab:
#   1. Kostenlos registrieren auf https://dashboard.ngrok.com/signup
#   2. Auth-Token kopieren von https://dashboard.ngrok.com/get-started/your-authtoken
#   3. In Colab links auf das 🔑-Icon "Secrets" -> "Add new secret":
#         Name:  NGROK_TOKEN
#         Value: (dein Token)
#      Notebook access an -> Speichern.
#   Danach reicht "Run all" - Token wird automatisch geladen.
%pip install -q pyngrok

from pyngrok import ngrok, conf

# Vorherige Tunnels desselben Prozesses aufraeumen (Cell-Rerun-safe).
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass

_token = None
try:
    from google.colab import userdata
    _token = userdata.get("NGROK_TOKEN")
except Exception:
    pass
if not _token:
    _token = os.environ.get("NGROK_TOKEN")
if not _token:
    # Hardcoded Fallback-Token fuer diesen Laptop (nicht committen).
    _token = "3HtsWbCZvpQuQVAYdx8yTMVVdAx_bAd2nWnDJcX5FeBgyJm1"

if not _token:
    raise RuntimeError(
        "❌ Kein NGROK_TOKEN gefunden.\n"
        "   1) Registriere dich kostenlos auf https://dashboard.ngrok.com/signup\n"
        "   2) Kopiere den Token von https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "   3) In Colab: 🔑 Secrets -> Add -> Name=NGROK_TOKEN, Value=<Token>, Access an.\n"
        "   4) Diese Zelle erneut ausfuehren."
    )

conf.get_default().auth_token = _token

# FastAPI im Hintergrund
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT,
                               log_level="warning", access_log=False),
    daemon=True,
).start()

for _ in range(30):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1).read()
        print(f"✅ FastAPI läuft auf Port {PORT}")
        break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("FastAPI-Start fehlgeschlagen.")

# ngrok HTTP-Tunnel -> https://XXXX.ngrok-free.app (WS + WSS out-of-the-box)
tunnel = ngrok.connect(PORT, proto="http", bind_tls=True)
public_url = tunnel.public_url

print("\n" + "=" * 68)
print(f"🌍 Backend-URL für die App:  {public_url}")
print("=" * 68)
print(f"Health-Check:  {public_url}/health")
print(f"Live-Logs:     {public_url}/logs")
print(f"WebSocket:     {public_url.replace('https://', 'wss://')}/stream")
print("\nTrage die Backend-URL in den Einstellungen der App ein.")


## 🔎 Backend-Logs anschauen

Diese Zelle jederzeit **erneut ausführen**, um die letzten Backend-Timings zu sehen.  
Alternativ im Handy-Browser: `{PUBLIC_URL}/logs`  (HTML mit Auto-Refresh).


In [ ]:
# --- Backend-Logs Live (jederzeit erneut ausfuehren) ---
import subprocess
out = subprocess.run(["tail", "-n", "40", "/content/backend.log"], capture_output=True, text=True)
print(out.stdout or "(noch keine Logs)")
if 'public_url' in dir():
    print(f"\n🌍 Live im Browser: {public_url}/logs")
